In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import time
import pandas as pd

In [16]:
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
INPUT_SIZE = 28 * 28
OUTPUT_SIZE = 10

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [18]:
class MNISTCSVDataset(Dataset):
    def __init__(self, csv_file, is_test=False):
        df = pd.read_csv(csv_file)
        
        if is_test and 'label' not in df.columns:
            self.X = df.values / 255.0
            self.y = None
        else:
            self.X = df.iloc[:, 1:].values / 255.0  
            self.y = df.iloc[:, 0].values            

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = torch.tensor(self.X[idx], dtype=torch.float32)
        
        if self.y is not None:
            label = torch.tensor(self.y[idx], dtype=torch.long)
            return image, label
        
        return image

train_dataset = MNISTCSVDataset("mnist/train.csv", is_test=False)
test_dataset = MNISTCSVDataset("mnist/test.csv", is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [19]:
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = x.reshape(-1, INPUT_SIZE)
        
        out = self.fc1(x)
        out = self.relu(out)
        
        out = self.fc2(out)
        return out

model = SimpleMLP(INPUT_SIZE, hidden_size=128, output_size=OUTPUT_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs):
    print("\n--- Starting Training ---")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, (images, labels) in enumerate(train_loader):
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")
    
    end_time = time.time()
    print(f"\nTraining finished in {end_time - start_time:.2f} seconds.")

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for images in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            predictions.extend(predicted.cpu().numpy())

    return predictions

In [ ]:
train_model(model, train_loader, criterion, optimizer, NUM_EPOCHS)


--- Starting Training ---
Epoch [1/10], Loss: 0.4769
Epoch [2/10], Loss: 0.2235
Epoch [3/10], Loss: 0.1681
Epoch [4/10], Loss: 0.1346
Epoch [5/10], Loss: 0.1099
Epoch [6/10], Loss: 0.0928
Epoch [7/10], Loss: 0.0782
Epoch [8/10], Loss: 0.0682
Epoch [9/10], Loss: 0.0581
Epoch [10/10], Loss: 0.0503

Training finished in 30.95 seconds.


In [23]:
print(evaluate_model(model, test_loader))

[2, 0, 9, 4, 3, 7, 0, 3, 0, 3, 5, 7, 4, 0, 4, 3, 3, 1, 9, 0, 9, 1, 1, 5, 7, 4, 2, 7, 4, 7, 7, 5, 4, 2, 6, 2, 5, 5, 1, 6, 7, 7, 4, 9, 8, 7, 8, 2, 6, 7, 6, 8, 8, 3, 8, 2, 1, 2, 2, 5, 4, 1, 7, 0, 0, 0, 1, 9, 0, 1, 6, 5, 8, 8, 2, 8, 3, 9, 2, 3, 5, 9, 1, 0, 9, 2, 4, 3, 6, 7, 2, 0, 6, 6, 1, 4, 3, 9, 7, 4, 0, 9, 2, 0, 7, 3, 0, 5, 0, 8, 0, 0, 4, 7, 1, 7, 1, 1, 3, 3, 3, 7, 2, 8, 6, 3, 8, 7, 8, 4, 3, 5, 6, 0, 0, 0, 3, 1, 5, 6, 5, 3, 4, 5, 5, 8, 7, 7, 2, 8, 4, 3, 5, 6, 5, 3, 7, 3, 7, 8, 3, 0, 4, 5, 1, 2, 7, 6, 3, 0, 2, 7, 8, 6, 1, 3, 7, 4, 1, 2, 4, 8, 5, 2, 4, 9, 2, 1, 6, 0, 6, 1, 4, 9, 6, 0, 9, 7, 6, 9, 1, 9, 0, 9, 9, 0, 8, 4, 6, 2, 0, 9, 3, 6, 3, 2, 1, 6, 3, 4, 2, 3, 1, 2, 2, 0, 4, 6, 1, 0, 0, 4, 9, 1, 7, 3, 2, 3, 8, 6, 8, 6, 2, 8, 5, 5, 4, 8, 3, 9, 9, 7, 1, 3, 8, 4, 5, 1, 4, 5, 6, 3, 3, 5, 7, 0, 6, 8, 3, 1, 6, 0, 6, 3, 9, 9, 1, 5, 8, 4, 0, 9, 2, 0, 5, 3, 7, 1, 9, 9, 5, 7, 7, 9, 9, 6, 3, 0, 3, 3, 6, 9, 8, 2, 6, 3, 7, 1, 4, 5, 8, 5, 9, 0, 0, 3, 8, 4, 1, 8, 4, 1, 1, 9, 8, 4, 5, 1, 5, 3, 6, 3, 1, 